<a href="https://colab.research.google.com/github/JulTob/BasesDeDatos/blob/main/%F0%9D%99%B9%F0%9D%9A%83_BasesDatos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🄱🄰🅂🄴🅂 🄳🄴 🄳🄰🅃🄾🅂**

# 𝙹𝚞𝚕𝚒𝚘 𝚃𝚘𝚋𝚘𝚜𝚘
### **2025**

---

###  Esquema metodológico de trabajo

Se introducen ejemplos básicos de SQL, paralelos a los operadores del álgebra relacional.

Las uniones de las tablas se tienen que realizar bien mediante el comando **WHERE** o, si fuera necesario, mediante los comandos **LEFT JOIN** o **RIGHT JOIN**, y **no con el uso del comando INNER JOIN**.

---

### Enunciado

Descargar la Base de Datos **“Human Resources” (HR)**, revisar atentamente el esquema relacional y realizar las consultas SQL.

---




Se ha usado el entorno Python para preparar a sqlite3, en una llamada funcional que importa los datos del entorno GitHub, después de haberlos pasado a un formato csv que permite una carga rápida y eficiente.

Las tablas han recibido nombres para el entorno SQL, e internamente se han guardado como bases de datos (dataframes) de NumPy. Esto solo afecta a que podemos usar la función display para enseñar el Head+Tail de los datos obtenidos por la Query. La ventaja es que tablas largas se colapsan automáticamente.

In [62]:
import pandas as pd
import sqlite3

base_url = "https://raw.githubusercontent.com/JulTob/Data/main/PD/HR/"

table_files = {
    "COUNTRIES":   "HR_countries.csv",
    "DEPARTMENTS": "HR_departments.csv",
    "EMPLOYEES":   "HR_employees.csv",
    "JOB_HISTORY": "HR_job_history.csv",
    "JOBS":        "HR_jobs.csv",
    "LOCATIONS":   "HR_locations.csv",
    "REGIONS":     "HR_regions.csv",
}

dfs = {}
for table_name, filename in table_files.items():
    url = base_url + filename
    dfs[table_name] = pd.read_csv(url)

# Create a SQLite in-memory database (or specify a file path like 'HR.sqlite')
conn = sqlite3.connect(":memory:") # Using ':memory:' for an in-memory database

for table_name, df in dfs.items():
    df.to_sql(table_name, conn, if_exists="replace", index=False)

print("SQLite database created with tables:")
# Query to check table names
query_tables = "SELECT name FROM sqlite_master WHERE type='table';"
display(pd.read_sql_query(query_tables, conn))

# The connection 'conn' is now open and contains your data, ready for queries
# We will keep the connection open for subsequent queries.

SQLite database created with tables:


,name
0,COUNTRIES
1,DEPARTMENTS
2,EMPLOYEES
3,JOB_HISTORY
4,JOBS
5,LOCATIONS
6,REGIONS


Se ha realizado una función que aisla el procedimiento interno, permitiendo centrarnos en las solicitudes de SQL.

Además, la función aporta invariablemente el tamaño de los datos solicitados, con un print del número de entradas, los datos por entrada, y el total de datos recogidos.

In [63]:

def SQL(sql_query):
    df_result = pd.read_sql_query(sql_query, conn)
    display(df_result)
    rows, cols = df_result.shape
    print(f"{rows}x{cols}: {rows * cols} data elements")


sql_query = """
Select *
From COUNTRIES;
"""


SQL(sql_query)

,country_id,country_name,region_id
0,AR,Argentina,2
1,AU,Australia,3
2,BE,Belgium,1
3,BR,Brazil,2
4,CA,Canada,2
5,CH,Switzerland,1
6,CN,China,3
7,DE,Germany,1
8,DK,Denmark,1
9,EG,Egypt,4


25x3: 75 data elements


In [64]:
SQL("""
Select *
From DEPARTMENTS;
""")

,department_id,department_name,manager_id,location_id
0,10,Administration,200.0,1700
1,20,Marketing,201.0,1800
2,30,Purchasing,114.0,1700
3,40,Human Resources,203.0,2400
4,50,Shipping,121.0,1500
5,60,IT,103.0,1400
6,70,Public Relations,204.0,2700
7,80,Sales,145.0,2500
8,90,Executive,100.0,1700
9,100,Finance,108.0,1700


27x4: 108 data elements


En este diagrama Podemos consultar la estructura ER simplificada. El tenerla en texto permite consultar nombres y copiarlos de manera sencilla para las *queries*.

```text
Legend:
⬜️ = Primary Key (Color matches table)
⚪️ = Foreign Key (Color matches related table)

Table Color Codes:
🟥 = REGIONS
🟧 = COUNTRIES
🟨 = LOCATIONS
🟩 = DEPARTMENTS
🟦 = EMPLOYEES
🟪 = JOBS
🟫 = JOB_HISTORY


┌───────────────────┐
│ 🟥     REGIONS    │
├───────────────────┤
│ 🟥 REGION_ID      │
│    REGION_NAME    │
└───────────────────┘

┌─────────────────────┐
│ 🟧     COUNTRIES    │
├─────────────────────┤
│ 🟧 COUNTRY_ID       │
│    COUNTRY_NAME     │
│ 🔴 REGION_ID        │ ⟵ | (1:M) has | ⟵  🟥 REGIONS
└─────────────────────┘

┌───────────────────────┐
│ 🟨     LOCATIONS      │
├───────────────────────┤
│ 🟨 LOCATION_ID        │
│    STREET_ADDRESS     │
│    POSTAL_CODE        │
│    CITY               │
│    STATE_PROVINCE     │
│ 🟠 COUNTRY_ID         │ ⟵ | (1:M) has | ⟵ 🟧 COUNTRIES
└───────────────────────┘

┌─────────────────────────┐
│ 🟩     DEPARTMENTS      │
├─────────────────────────┤
│ 🟩 DEPARTMENT_ID        │
│    DEPARTMENT_NAME      │
│ 🔵 MANAGER_ID           │ ⟵ | (1:M) manages | ⟵ 🟦EMPLOYEES
│ 🟡 LOCATION_ID          │ ⟵ | (1:M) hosts | ⟵ 🟨LOCATIONS
└─────────────────────────┘

┌─────────────────────┐
│ 🟦     EMPLOYEES    │
├─────────────────────┤
│ 🟦 EMPLOYEE_ID      │
│    FIRST_NAME       │
│    LAST_NAME        │
│    EMAIL            │
│    PHONE_NUMBER     │
│    HIRE_DATE        │
│ 🟣 JOB_ID           │ ⟵ | (1:M) assigned to | ⟵ 🟪JOBS
│    SALARY           │
│    COMMISSION_PCT   │
│ 🔵 MANAGER_ID       │ ⟵ | (1:M) supervises | ⟵ 🟦EMPLOYEES
│ 🟢 DEPARTMENT_ID    │ ⟵ | (1:M) belongs to | ⟵ 🟩DEPARTMENTS
└─────────────────────┘

┌─────────────────────┐
│ 🟫   JOB_HISTORY    │
├─────────────────────┤
│ 🟫🔵 EMPLOYEE_ID     │ ⟵ | (1:M) has history | ⟵ 🟦EMPLOYEES
│ 🟫 START_DATE       │
│    END_DATE         │
│ 🟣 JOB_ID           │ ⟵ | (1:M) recorded in | ⟵ 🟪JOBS
│ 🟢 DEPARTMENT_ID    │ ⟵ | (1:M) includes | ⟵ 🟩DEPARTMENTS
└─────────────────────┘

┌────────────────┐
│ 🟪     JOBS    │
├────────────────┤
│ 🟪 JOB_ID      │
│    JOB_TITLE   │
│    MIN_SALARY  │
│    MAX_SALARY  │
└────────────────┘
```

## Ejercicio a)
Mostrar el Nombre, Apellidos y número de teléfono de los empleados cuyo trabajo sea de vendedor (es decir que job_id sea IT_PROG).

1. ## $R_1$ := 𝜎(JOB_ID = "IT_PROG")[EMPLOYEES]
2. ## $R_2$ := 𝜋(FIRST_NAME, LAST_NAME, PHONE_NUMBER)[$R_1$]

In [65]:
SQL("""
SELECT
    first_name as Nombre,
    last_name as Apellidos,
    phone_number as Telefono
FROM EMPLOYEES
WHERE JOB_ID = 'IT_PROG'
;
""")

,Nombre,Apellidos,Telefono
0,Alexander,Hunold,590.423.4567
1,Bruce,Ernst,590.423.4568
2,David,Austin,590.423.4569
3,Valli,Pataballa,590.423.4560
4,Diana,Lorentz,590.423.5567


5x3: 15 data elements


## Ejercicio b)
Dar una lista con el Nombre y Apellidos de Empleados que no tengan comisión asignada (COMMISSION_PCT).

1. 𝜎(COMMISSION_PCT is ␀)[EMPLOYEES]
2. 𝜋(FIRST_NAME, LAST_NAME)[1.]

In [66]:
SQL("""
SELECT
    first_name as Nombre,
    last_name as Apellidos
FROM EMPLOYEES
WHERE  COMMISSION_PCT IS 0
    OR COMMISSION_PCT IS NULL
;
""")

,Nombre,Apellidos
0,Steven,King
1,Neena,Kochhar
2,Lex,De Haan
3,Alexander,Hunold
4,Bruce,Ernst
...,...,...
102,Pat,Fay
103,Susan,Mavris
104,Hermann,Baer
105,Shelley,Higgins


107x2: 214 data elements


## Ejercicio c)
Obtener una lista donde se indique cuantos empleados tiene comisión y cuantos no.

> Nota: El planteamiento tiene cierta ambiguedad. Por un lado en entorno técnico, entenderíamos que son los valores nulos en comisiones, _peeeero_ en entornos laborales, como en ofimática, esta pregunta solicitaría también aquellos que tienen un valor de 0 en el campo no-nulo. Es interesante investigar ambos planteamientos. Voy a tomar esta ruta dual.

1. E := 𝜌[EMPLOYEE]
2. A := 𝜎(COMISSION is ␀)[E]
2. B := 𝜎(COMISSION is 0)[E]
2. C := 𝜎(COMISSION > 0)[E]
3. ⏎ R := |A| ∪ |B| ∪ |C|

In [67]:
SQL("""
SELECT
    COUNT(*) AS TOTAL,
    COUNT(COMMISSION_PCT) AS CON_COMISION,
    COUNT(*) - COUNT(COMMISSION_PCT) AS SIN_COMISION
FROM EMPLOYEES;
""")

,TOTAL,CON_COMISION,SIN_COMISION
0,107,35,72


1x3: 3 data elements


Truco: Se puede hacer un **histograma** analítico de los datos con la función de COUNT aplicada a un GROUP-BY. Esto separa los tipos de la columna, con cada count aplicada a cada grupo en lugar de al total de tabla.

In [116]:
SQL("""
SELECT
    COMMISSION_PCT as Comision,
    COUNT(*) as Conteo
FROM EMPLOYEES
GROUP BY COMMISSION_PCT
;
""")

,Comision,Conteo
0,NaN,72
1,0.0,35


2x2: 4 data elements


Esta solución la marco como _Truco_ porque en otros contextos o en valores continuos de Commission_PCT podría potencialmente crear una tabla enorme. Aunque también podríamos extraer la información de esa tabla con `where`.

Alternativamente, la forma más meticulosa sería así:

In [70]:
SQL("""
SELECT
    COUNT(COMMISSION_PCT) as "Known",
    SUM(COMMISSION_PCT IS 0) as "Comission=0",
    SUM(COMMISSION_PCT IS NOT NULL
        AND COMMISSION_PCT IS NOT 0) as "Comission>0",
    SUM(COMMISSION_PCT IS NULL) as "Unknown",
    SUM(COMMISSION_PCT) as "Total Comissions"

FROM EMPLOYEES
;
""")

,Known,Comission=0,Comission>0,Unknown,Total Comissions
0,35,35,0,72,0.0


1x5: 5 data elements


> Tenemos que usar SUM(condición), donde cada entrada evalua la condición en esa instancia. En caso afirmativo devuelve 1, con lo que se suma al tallaje total.

> PD: Sabemos que solo podemos tener comisiones >0 por lógica del negocio (Bussiness Logic).

In [69]:
SQL("""
SELECT
    "Sin Comision" as Tipo,
    COUNT(*) as Total FROM EMPLOYEES
        WHERE COMMISSION_PCT IS NULL
UNION SELECT
    "Con Comision" as Tipo,
    COUNT(*) as Total FROM EMPLOYEES
    WHERE COMMISSION_PCT IS NOT NULL
;
""")

,Tipo,Total
0,Con Comision,35
1,Sin Comision,72


2x2: 4 data elements


## Ejercicio d)
Obtener una lista de empleados con el nombre, los apellidos, el email y el salario de cada uno éstos ordenada de menor a mayor salario.

$$
𝜗:=\text{FIRST_NAME},\ \text{LAST_NAME},\ \text{EMAIL},\ \text{SALARY}
$$
$$
R₁ :=
𝜋_{𝜗}
[\text{EMPLOYEES}]
$$

$$
𝜏_{\text{SALARY}↑}[R_1]
$$

In [71]:
SQL("""
SELECT
    first_name AS Nombre,
    last_name AS Apellidos,
    email AS Email,
    salary AS Salario
FROM EMPLOYEES
ORDER BY SALARY
;
""")

,Nombre,Apellidos,Email,Salario
0,TJ,Olson,TJOLSON,2100
1,Steven,Markle,SMARKLE,2200
2,Hazel,Philtanker,HPHILTAN,2200
3,James,Landry,JLANDRY,2400
4,Ki,Gee,KGEE,2400
...,...,...,...,...
102,Karen,Partners,KPARTNER,13500
103,John,Russell,JRUSSEL,14000
104,Neena,Kochhar,NKOCHHAR,17000
105,Lex,De Haan,LDEHAAN,17000


107x4: 428 data elements


## Ejercicio e)
Realizar una lista como la anterior añadiendo además la columna correspondiente al nombre del departamento. ¿Qué se observa respecto al número de filas obtenidas en esta consulta respecto a la anterior? ¿Qué explicación puedes dar al respecto?

$$
E:= 𝛒(EMPLOYEES)
$$
$$
D:= 𝛒(DEPARTMENTS)
$$

$$
a:= E.Department＿ID = D.Department＿ID
$$

$$
ED :=
E \underset{a}{⋈} D
$$

$$
b :=
{
FIRST＿NAME,
LAST＿NAME,
EMAIL,
SALARY,
DEPARTMENT＿NAME
}
$$

$$
R₁ := 𝜋_{(b)}[ED]
$$

$$
R :=
\tau_{\text{SALARY} \uparrow}(R_1)
$$


Tú decides el siguiente paso.


In [72]:
SQL("""
SELECT
    E.first_name AS Nombre,
    E.last_name AS Apellidos,
    E.email AS Email,
    E.salary AS Salario,
    D.department_name AS Departamento
FROM EMPLOYEES AS E
    JOIN DEPARTMENTS AS D
        ON E.DEPARTMENT_ID = D.DEPARTMENT_ID
ORDER BY
    E.salary
;
""")

,Nombre,Apellidos,Email,Salario,Departamento
0,TJ,Olson,TJOLSON,2100,Shipping
1,Steven,Markle,SMARKLE,2200,Shipping
2,Hazel,Philtanker,HPHILTAN,2200,Shipping
3,James,Landry,JLANDRY,2400,Shipping
4,Ki,Gee,KGEE,2400,Shipping
...,...,...,...,...,...
101,Karen,Partners,KPARTNER,13500,Sales
102,John,Russell,JRUSSEL,14000,Sales
103,Neena,Kochhar,NKOCHHAR,17000,Executive
104,Lex,De Haan,LDEHAAN,17000,Executive


106x5: 530 data elements


Hay un miembro menos. El inner join no ha encontrado match

Vamos a verificar cuántos empleados tienen un `DEPARTMENT_ID` nulo, lo cual explicaría por qué no fueron incluidos en el `INNER JOIN`.

In [73]:
SQL("""
SELECT
    COUNT(*) AS Employees_Without_Department
FROM
    EMPLOYEES
WHERE
    DEPARTMENT_ID IS NULL;
""")

,Employees_Without_Department
0,1


1x1: 1 data elements


Efectivamente, la ausencia de este empleado indica un Empleado sin departamento, omitido por el inner join.

Un método sin usar el comando `INNER JOIN` sería usando `FROM-WHERE`

In [74]:
SQL("""
SELECT
    EMPLOYEES.first_name,
    EMPLOYEES.Department_ID,
    DEPARTMENTS.Department_ID,
    DEPARTMENTS.Department_Name
FROM
    EMPLOYEES, DEPARTMENTS
WHERE
    EMPLOYEES.Department_ID = DEPARTMENTS.Department_ID
    ;
""")

,first_name,department_id,department_id,department_name
0,Steven,90.0,90,Executive
1,Neena,90.0,90,Executive
2,Lex,90.0,90,Executive
3,Alexander,60.0,60,IT
4,Bruce,60.0,60,IT
...,...,...,...,...
101,Pat,20.0,20,Marketing
102,Susan,40.0,40,Human Resources
103,Hermann,70.0,70,Public Relations
104,Shelley,110.0,110,Accounting


106x4: 424 data elements


In [75]:
SQL("""
SELECT
    EMPLOYEES.first_name AS Nombre,
    EMPLOYEES.last_name AS Apellidos,
    EMPLOYEES.email AS Email,
    EMPLOYEES.salary AS Salario,
    DEPARTMENTS.Department_Name
FROM
    EMPLOYEES, DEPARTMENTS
WHERE
    EMPLOYEES.Department_ID = DEPARTMENTS.Department_ID
    ;
""")

,Nombre,Apellidos,Email,Salario,department_name
0,Steven,King,SKING,24000,Executive
1,Neena,Kochhar,NKOCHHAR,17000,Executive
2,Lex,De Haan,LDEHAAN,17000,Executive
3,Alexander,Hunold,AHUNOLD,9000,IT
4,Bruce,Ernst,BERNST,6000,IT
...,...,...,...,...,...
101,Pat,Fay,PFAY,6000,Marketing
102,Susan,Mavris,SMAVRIS,6500,Human Resources
103,Hermann,Baer,HBAER,10000,Public Relations
104,Shelley,Higgins,SHIGGINS,12000,Accounting


106x5: 530 data elements


## Exploración Propia: Contar Empleados por Departamento

Quiero probar a obtener el nombre del departamento y el número de empleados que pertenecen a cada uno. Sería conveniente para entender el funcionamiento de los join.

In [76]:
SQL("""
SELECT
    Dep.department_name AS NombreDepartamento,
    COUNT(Emp.employee_id) AS NumeroDeEmpleados
FROM
    DEPARTMENTS AS Dep
INNER JOIN
    EMPLOYEES AS Emp
    ON
        Dep.department_id = Emp.department_id
GROUP BY
    Dep.department_name
ORDER BY
    NumeroDeEmpleados DESC
;
""")
# Algorithm actions

,NombreDepartamento,NumeroDeEmpleados
0,Shipping,45
1,Sales,34
2,Purchasing,6
3,Finance,6
4,IT,5
5,Executive,3
6,Marketing,2
7,Accounting,2
8,Public Relations,1
9,Human Resources,1


11x2: 22 data elements


## Ejercicio f)
Obtener una lista de aquellos apellidos de empleados que no comiencen por la letra e ni terminen por la letra n, pero que contengan la letra t ordenada alfabéticamente por apellido de la A a la Z.

In [77]:
SQL("""
SELECT DISTINCT
    last_name AS Apellido
FROM
    EMPLOYEES
WHERE
    last_name NOT LIKE 'E%'
    AND last_name NOT LIKE '%n'
    AND last_name LIKE '%t%'
ORDER BY
    last_name ASC;
""")

,Apellido
0,Bates
1,Bissot
2,Cambrault
3,Faviet
4,Gates
5,Gietz
6,Grant
7,Lorentz
8,Matos
9,Partners


20x1: 20 data elements


## Ejercicio g)
Mostrar Nombre y Apellidos en una única columna con el formato 'Apellidos, Nombre' para todos los empleados cuyo teléfono contenga el prefijo 515.

In [78]:
SQL("""
SELECT
    last_name || ', ' || first_name AS "Nombre Completo"
FROM
    EMPLOYEES
WHERE
    phone_number LIKE '515%'
;
""")

# NOTA: EN ACCESS concatenar Sería
# (E.last_name & ', ' & E.first_name) AS NombreCompleto


,Nombre Completo
0,"King, Steven"
1,"Kochhar, Neena"
2,"De Haan, Lex"
3,"Greenberg, Nancy"
4,"Faviet, Daniel"
5,"Chen, John"
6,"Sciarra, Ismael"
7,"Urman, Jose Manuel"
8,"Popp, Luis"
9,"Raphaely, Den"


21x1: 21 data elements


## Ejercicio h)
Obtener una lista de aquellos apellidos de empleados que están repetidos y no terminen por la letra h o r, indicando el número de repeticiones.

In [79]:
SQL("""
SELECT
    last_name AS Apellidos,
    COUNT(last_name) AS Repeticiones
FROM EMPLOYEES
WHERE
    last_name NOT LIKE '%h'
    AND
    last_name NOT LIKE '%r'
GROUP BY
    last_name
HAVING
    Repeticiones > 1
;
""")

,Apellidos,Repeticiones
0,Cambrault,2
1,Grant,2
2,King,2


3x2: 6 data elements


## Ejercicio i)
Obtener una lista con los apellidos, email y nombre del departamento de los empleados directores de departamento ordenada por el nombre del departamento del cual sean director.

$$
A := ｛ Apellidos, Email, Departamento｝
$$
$$
E := 𝛒(EMPLOYEES)
$$
$$
D := 𝛒(DEPARTMENTS)
$$
$$
𝛼 := (E.EMPLOYEE＿ID = D.MANAGER＿ID)
$$
$$
ED :=
  E
  \underset{𝛼 }{⋈}
  D
$$
$$
R₁ :=
𝜋(A)[ED]
$$
$$
𝜏(DEPARTMENT_NAME↑)[R₁]
$$

In [80]:
SQL("""
SELECT
    E.LAST_NAME AS Apellidos,
    E.EMAIL AS Email,
    D.DEPARTMENT_NAME AS Departamento
FROM EMPLOYEES AS E,
    DEPARTMENTS AS D
WHERE
    E.EMPLOYEE_ID = D.MANAGER_ID
ORDER BY
    D.DEPARTMENT_NAME
;
""")

,Apellidos,Email,Departamento
0,Higgins,SHIGGINS,Accounting
1,Whalen,JWHALEN,Administration
2,King,SKING,Executive
3,Greenberg,NGREENBE,Finance
4,Mavris,SMAVRIS,Human Resources
5,Hunold,AHUNOLD,IT
6,Hartstein,MHARTSTE,Marketing
7,Baer,HBAER,Public Relations
8,Raphaely,DRAPHEAL,Purchasing
9,Russell,JRUSSEL,Sales


11x3: 33 data elements


## Ejercicio j)
Obtener una lista con los Nombres y Apellidos de los directores de departamento que superen el salario medio de todos los directores de departamento de la empresa, indicando también el nombre de la región dónde el departamento está ubicado.

In [81]:
SQL("""
SELECT
    E.FIRST_NAME AS Nombre,
    E.LAST_NAME AS Apellidos,
    D.DEPARTMENT_NAME AS Departamento,
    R.REGION_NAME AS Region
FROM EMPLOYEES AS E,
    DEPARTMENTS AS D,
    LOCATIONS AS L,
    COUNTRIES AS C,
    REGIONS AS R
WHERE
    E.EMPLOYEE_ID = D.MANAGER_ID
    AND
    D.LOCATION_ID = L.LOCATION_ID
    AND
    L.COUNTRY_ID = C.COUNTRY_ID
    AND
    C.REGION_ID = R.REGION_ID
    AND
    E.SALARY > (SELECT AVG(E2.SALARY)
        FROM EMPLOYEES AS E2, DEPARTMENTS AS D2
        WHERE E2.EMPLOYEE_ID = D2.MANAGER_ID)
ORDER BY
    D.DEPARTMENT_NAME
;
""")

,Nombre,Apellidos,Departamento,Region
0,Shelley,Higgins,Accounting,Americas
1,Steven,King,Executive,Americas
2,Nancy,Greenberg,Finance,Americas
3,Michael,Hartstein,Marketing,Americas
4,John,Russell,Sales,Europe


5x4: 20 data elements


## Ejercicio k)
Obtener una lista con los nombres de los departamentos con las Ciudades, Nombre de País y Nombre de la Región a donde pertenezcan ordenadas alfabéticamente por el nombre del país de la Z a la A.

In [82]:
SQL("""
SELECT
    D.Department_Name AS Departamento,
    L.City AS Ciudad,
    C.Country_Name AS Pais,
    R.Region_Name AS Region
FROM
    DEPARTMENTS AS D,
    LOCATIONS AS L,
    COUNTRIES AS C,
    REGIONS AS R
WHERE
    D.Location_ID = L.Location_ID
    AND
    L.Country_ID = C.Country_ID
    AND
    C.Region_ID = R.Region_ID

ORDER BY C.COUNTRY_NAME DESC
;
""")

,Departamento,Ciudad,Pais,Region
0,Administration,Seattle,United States of America,Americas
1,Purchasing,Seattle,United States of America,Americas
2,Shipping,South San Francisco,United States of America,Americas
3,IT,Southlake,United States of America,Americas
4,Executive,Seattle,United States of America,Americas
5,Finance,Seattle,United States of America,Americas
6,Accounting,Seattle,United States of America,Americas
7,Treasury,Seattle,United States of America,Americas
8,Corporate Tax,Seattle,United States of America,Americas
9,Control And Credit,Seattle,United States of America,Americas


27x4: 108 data elements


## Ejercicio l)
Obtener una lista con los nombres y apellidos de los empleados, así como el nombre de su categoría laboral (JOB_TITLE) y el salario mínimo de ésta, ordenado de menor a mayor salario mínimo.

1. EJ := E⋈J (E.Job_ID=J.Job_ID)
2. 𝜋(Nombres, Apellidos, Puesto, Salario_Minimo)[EJ]
3. 𝜏 [2.] (Salario_Minimo↑)

In [83]:
SQL("""
SELECT
    E.First_Name AS Nombre,
    E.Last_Name AS Apellidos,
    J.Job_Title AS Puesto,
    J.Min_Salary AS "Salario Mínimo"
FROM
    EMPLOYEES AS E,
    JOBS AS J
WHERE E.JOB_ID = J.JOB_ID
ORDER BY J.MIN_SALARY ASC
;
""")

,Nombre,Apellidos,Puesto,Salario Mínimo
0,Julia,Nayer,Stock Clerk,2000
1,Irene,Mikkilineni,Stock Clerk,2000
2,James,Landry,Stock Clerk,2000
3,Steven,Markle,Stock Clerk,2000
4,Laura,Bissot,Stock Clerk,2000
...,...,...,...,...
102,Gerald,Cambrault,Sales Manager,10000
103,Eleni,Zlotkey,Sales Manager,10000
104,Neena,Kochhar,Administration Vice President,15000
105,Lex,De Haan,Administration Vice President,15000


107x4: 428 data elements


## Ejercicio m)
Obtener una lista de categoría laborales indicando: el nombre, el número de empleados que pertenecen, el mayor salario pagado, el menor salario pagado y el salario medio todo ello referente a la mencionada categoría laboral ordenada de mayor a menor salario medio.

1. E := 𝜌(EMPLOYEES)
2. J := 𝜌(JOBS)
3. EJ := E⋈J(E.JOB_ID=J.JOB_ID)
1. Categorías := 𝛾{J.Job_Title,|E.Job_ID|, J.Max_Salario, J.Min_Salario, AVG(E.Salario)[EJ]}
2. R₁ := 𝜋(Puesto, N_Empleados,  Max_Salario, Min_Salario, Med_Salario)[Categorías]
3. R := 𝜏 (Med_Salario↓)[R₁]


In [84]:
SQL("""
SELECT
    J.Job_Title AS Puesto,
    COUNT(E.Job_ID) as "Número de Empleados",
    J.MAX_SALARY AS "Salario Máximo",
    J.MIN_SALARY AS "Salario Mínimo",
    AVG(E.Salary) as "Salario Medio"
FROM
    JOBS AS J,
    EMPLOYEES AS E
WHERE
    J.JOB_ID = E.JOB_ID
GROUP BY
    J.JOB_ID
ORDER BY
    "Salario Medio" DESC
;""")

,Puesto,Número de Empleados,Salario Máximo,Salario Mínimo,Salario Medio
0,President,1,40000,20000,24000.0
1,Administration Vice President,2,30000,15000,17000.0
2,Marketing Manager,1,15000,9000,13000.0
3,Sales Manager,5,20000,10000,12200.0
4,Finance Manager,1,16000,8200,12000.0
5,Accounting Manager,1,16000,8200,12000.0
6,Purchasing Manager,1,15000,8000,11000.0
7,Public Relations Representative,1,10500,4500,10000.0
8,Sales Representative,30,12000,6000,8350.0
9,Public Accountant,1,9000,4200,8300.0


19x5: 95 data elements


Me preguntaba también si sería posible hacerlo con los datos reales de max/min en la tabla:

In [85]:
SQL("""
SELECT
    J.JOB_TITLE AS Puesto,
    COUNT(E.JOB_ID) AS NumeroEmpleados,
    MAX(E.SALARY) AS SalarioMaximo,
    MIN(E.SALARY) AS SalarioMinimo,
    AVG(E.SALARY) AS SalarioMedio
FROM
    JOBS AS J,
    EMPLOYEES AS E
WHERE
    J.JOB_ID = E.JOB_ID
GROUP BY
    J.JOB_ID, J.JOB_TITLE
ORDER BY
    SalarioMedio DESC;
""")

,Puesto,NumeroEmpleados,SalarioMaximo,SalarioMinimo,SalarioMedio
0,President,1,24000,24000,24000.0
1,Administration Vice President,2,17000,17000,17000.0
2,Marketing Manager,1,13000,13000,13000.0
3,Sales Manager,5,14000,10500,12200.0
4,Accounting Manager,1,12000,12000,12000.0
5,Finance Manager,1,12000,12000,12000.0
6,Purchasing Manager,1,11000,11000,11000.0
7,Public Relations Representative,1,10000,10000,10000.0
8,Sales Representative,30,11500,6100,8350.0
9,Public Accountant,1,8300,8300,8300.0


19x5: 95 data elements


He visto que funciona igual en `GROUP BY` si se selecciona cualquiera de los `J.JOB_ID, J.JOB_TITLE` o ambos. Me ha parecido interesante porque ambos pueden identificar el Trabajo, pero se podría granular varios modelos de Job_Title similares, o bajo el mismo nombre, pero con descriptivos diferentes. Esto puede ser util si se tiene un grupo de trabajo comercial orientado al público y otro orientado a empresas, por ejemplo.

## Ejercicio n)
Listar los nombres de todos los departamentos y el nombre de su director, incluyendo los departamentos que no tienen director asignado, en este último caso la columna dirá ‘Sin Director Asignado’.

* D := 𝜌(DEPARTMENTS)
* E := 𝜌(EMPLOYEES)
* 𝛼 := D.MANAGER_ID = E.EMPLOYEE_ID
* R₁ := D ⋈ E (𝛼)
* P₁ := 𝜋(
    - D.DEPARTMENT_NAME,
    - E.FIRST_NAME,
    - E.LAST_NAME)
    - [R₁]
* S := 𝜎(D.MANAGER_ID = ␀)[P₁]
* P₂ := 𝜋(
    - D.DEPARTMENT_NAME,
    - "Sin Director Asignado",
    - "-")
    - [S]
* R := P₁ ∪ P₂

  

In [86]:
SQL("""
SELECT DISTINCT
    D.DEPARTMENT_NAME AS Departamento,
    E.FIRST_NAME AS "Nombre Director",
    E.LAST_NAME AS "Apellidos Director"
FROM DEPARTMENTS AS D,
    EMPLOYEES AS E
WHERE
    D.MANAGER_ID = E.EMPLOYEE_ID
UNION
SELECT
    D2.DEPARTMENT_NAME AS Departamento,
    "Sin Director Asignado" AS "Nombre Director",
    "-" AS "Apellidos Director"
FROM DEPARTMENTS AS D2
WHERE D2.MANAGER_ID IS NULL
ORDER BY "Departamento"
;

""")

,Departamento,Nombre Director,Apellidos Director
0,Accounting,Shelley,Higgins
1,Administration,Jennifer,Whalen
2,Benefits,Sin Director Asignado,-
3,Construction,Sin Director Asignado,-
4,Contracting,Sin Director Asignado,-
5,Control And Credit,Sin Director Asignado,-
6,Corporate Tax,Sin Director Asignado,-
7,Executive,Steven,King
8,Finance,Nancy,Greenberg
9,Government Sales,Sin Director Asignado,-


27x3: 81 data elements


## Ejercicio o)
Obtener una lista con los nombres y apellidos de los empleados, así como el nombre y apellidos de su supervisor (mánager) ordenada alfabéticamente por el código de la categoría laboral (JOB_ID) de este último.

In [87]:
SQL("""
SELECT
    E.First_Name AS Nombre,
    E.Last_Name AS Apellidos,
    M.First_Name AS "Nombre Supervisor",
    M.Last_Name AS "Apellidos Supervisor"
FROM
    EMPLOYEES as E,
    EMPLOYEES as M
WHERE
    E.MANAGER_ID = M.EMPLOYEE_ID
ORDER BY
    M.JOB_ID
;
""")

,Nombre,Apellidos,Nombre Supervisor,Apellidos Supervisor
0,William,Gietz,Shelley,Higgins
1,Neena,Kochhar,Steven,King
2,Lex,De Haan,Steven,King
3,Den,Raphaely,Steven,King
4,Matthew,Weiss,Steven,King
...,...,...,...,...
101,Vance,Jones,Shanta,Vollman
102,Alana,Walsh,Kevin,Mourgos
103,Kevin,Feeney,Kevin,Mourgos
104,Donald,OConnell,Kevin,Mourgos


106x4: 424 data elements


## Ejercicio p)
Obtener una lista con los nombres de los departamentos y la ciudad donde éste está ubicado incluyendo el importe total de salarios que ese departamento paga a sus empleados.

-

In [88]:
SQL("""
SELECT
    D.DEPARTMENT_NAME AS Departamento,
    L.CITY AS Ciudad,
    SUM(E.SALARY) AS "Salario Total"
FROM
    DEPARTMENTS AS D,
    LOCATIONS AS L,
    EMPLOYEES AS E
WHERE
    D.LOCATION_ID = L.LOCATION_ID
    AND
    D.DEPARTMENT_ID = E.DEPARTMENT_ID
GROUP BY
    D.DEPARTMENT_ID
;
""")

,Departamento,Ciudad,Salario Total
0,Administration,Seattle,4400
1,Marketing,Toronto,19000
2,Purchasing,Seattle,24900
3,Human Resources,London,6500
4,Shipping,South San Francisco,156400
5,IT,Southlake,28800
6,Public Relations,Munich,10000
7,Sales,Oxford,304500
8,Executive,Seattle,58000
9,Finance,Seattle,51600


11x3: 33 data elements


## Ejercicio q)
Seleccionar los empleados que no sean supervisores de nadie dando sus Nombre y Apellidos separados por coma, así como el nombre la categoría laboral (JOB_TITLE) a la que pertenecen.

In [89]:
# Prueba de la Intermedia Para los Managers
SQL("""
SELECT
        DISTINCT E2.MANAGER_ID
        FROM
            EMPLOYEES AS E2
        WHERE
            E2.MANAGER_ID IS NOT NULL
""")

,manager_id
0,100.0
1,102.0
2,103.0
3,101.0
4,108.0
5,114.0
6,120.0
7,121.0
8,122.0
9,123.0


18x1: 18 data elements


In [90]:
SQL("""
SELECT
    E.Last_Name || ", " || E.First_Name As NombreCompleto,
    J.Job_Title As Puesto
FROM
    EMPLOYEES AS E,
    JOBS AS J
WHERE
    E.JOB_ID = J.JOB_ID
    AND E.EMPLOYEE_ID NOT IN (SELECT
        DISTINCT E2.MANAGER_ID
        FROM
            EMPLOYEES AS E2
        WHERE
            E2.MANAGER_ID IS NOT NULL

        )
;
""")

,NombreCompleto,Puesto
0,"Ernst, Bruce",Programmer
1,"Austin, David",Programmer
2,"Pataballa, Valli",Programmer
3,"Lorentz, Diana",Programmer
4,"Faviet, Daniel",Accountant
...,...,...
84,"Whalen, Jennifer",Administration Assistant
85,"Fay, Pat",Marketing Representative
86,"Mavris, Susan",Human Resources Representative
87,"Baer, Hermann",Public Relations Representative


89x2: 178 data elements


## Ejercicio r)
Obtener una lista con los nombres de departamento, el número de empleados, el salario máximo, el mínimo, así como el promedio de cada uno de estos departamentos, ordenados según el salario medio de menor a mayor.

In [91]:
SQL("""
SELECT
    D.DEPARTMENT_NAME AS Departamento,
    COUNT(E.EMPLOYEE_ID) AS "Número de Empleados",
    MAX(E.SALARY) AS "Salario Máximo",
    MIN(E.SALARY) AS "Salario Mínimo",
    ROUND(AVG(E.SALARY),2) AS "Salario Medio"
FROM
    DEPARTMENTS AS D,
    EMPLOYEES AS E
    WHERE
        D.DEPARTMENT_ID = E.DEPARTMENT_ID
    GROUP BY
        D.DEPARTMENT_ID
""")

,Departamento,Número de Empleados,Salario Máximo,Salario Mínimo,Salario Medio
0,Administration,1,4400,4400,4400.00
1,Marketing,2,13000,6000,9500.00
2,Purchasing,6,11000,2500,4150.00
3,Human Resources,1,6500,6500,6500.00
4,Shipping,45,8200,2100,3475.56
5,IT,5,9000,4200,5760.00
6,Public Relations,1,10000,10000,10000.00
7,Sales,34,14000,6100,8955.88
8,Executive,3,24000,17000,19333.33
9,Finance,6,12000,6900,8600.00


11x5: 55 data elements


## Ejercicio s)
Obtener el número de empleados contratados entre los años 1995 y 1999 que estén asignados a algún departamento.

In [92]:
SQL("""
SELECT
    HIRE_DATE AS Fecha
FROM EMPLOYEES
WHERE HIRE_DATE LIKE("%199[5-9]%")
""")

,Fecha


0x1: 0 data elements


In [93]:
SQL("""
SELECT
    count(HIRE_DATE) AS "Número de Empleados"
FROM EMPLOYEES
WHERE
    (   HIRE_DATE LIKE("%1999%")
    OR
        HIRE_DATE LIKE("%1998%")
    OR
        HIRE_DATE LIKE("%1997%")
    OR
        HIRE_DATE LIKE("%1996%")
    OR
        HIRE_DATE LIKE("%1995%")
    ) AND
        DEPARTMENT_ID IS NOT NULL
""")

,Número de Empleados
0,82


1x1: 1 data elements


## Ejercicio t)
Obtener el número de ciudades que tiene cada país, indicando el nombre de ésta.

In [94]:
SQL("""
SELECT
    C.COUNTRY_NAME AS Pais,
    COUNT(L.CITY) AS "Número de Ciudades"
FROM
    COUNTRIES AS C,
    LOCATIONS AS L
    WHERE
        C.COUNTRY_ID = L.COUNTRY_ID
    GROUP BY
        C.COUNTRY_NAME
    ORDER BY
        "Número de Ciudades" DESC
""")

,Pais,Número de Ciudades
0,United States of America,4
1,United Kingdom,3
2,Switzerland,2
3,Japan,2
4,Italy,2
5,Canada,2
6,Singapore,1
7,Netherlands,1
8,Mexico,1
9,India,1


14x2: 28 data elements


## Ejercicio u)
Obtener una lista de Ciudades junto al Nombre de País y Nombre de la Región a donde pertenezcan, en la cual se indique el número de departamentos que ésta aloja. La lista ha de estar ordenada de menor a mayor número de departamentos.

In [95]:
SQL("""
SELECT
    L.CITY AS Ciudad,
    C.COUNTRY_NAME AS Pais,
    R.REGION_NAME AS Region,
    COUNT(D.DEPARTMENT_ID) AS "Número de Departamentos"
FROM
    LOCATIONS AS L,
    COUNTRIES AS C,
    REGIONS AS R,
    DEPARTMENTS AS D
WHERE
    L.COUNTRY_ID = C.COUNTRY_ID
    AND
    C.REGION_ID = R.REGION_ID
    AND
    L.LOCATION_ID = D.LOCATION_ID
GROUP BY
    L.CITY
ORDER BY
    "Número de Departamentos"
;
""")

,Ciudad,Pais,Region,Número de Departamentos
0,London,United Kingdom,Europe,1
1,Munich,Germany,Europe,1
2,Oxford,United Kingdom,Europe,1
3,South San Francisco,United States of America,Americas,1
4,Southlake,United States of America,Americas,1
5,Toronto,Canada,Americas,1
6,Seattle,United States of America,Americas,21


7x4: 28 data elements


## Ejercicio v)
Obtener la lista de Ciudades en las que no se aloja ningún departamento de la empresa incluyendo el Nombre de País y Nombre de la Región.

In [96]:
SQL("""
SELECT
    L.CITY AS Ciudad,
    C.COUNTRY_NAME AS Pais,
    R.REGION_NAME AS Region
FROM
    LOCATIONS AS L,
    COUNTRIES AS C,
    REGIONS AS R
WHERE
    L.COUNTRY_ID = C.COUNTRY_ID
    AND
    C.REGION_ID = R.REGION_ID
    AND
    L.LOCATION_ID NOT IN (SELECT
        D2.LOCATION_ID
        FROM
            DEPARTMENTS AS D2)
""")

,Ciudad,Pais,Region
0,Roma,Italy,Europe
1,Venice,Italy,Europe
2,Tokyo,Japan,Asia
3,Hiroshima,Japan,Asia
4,South Brunswick,United States of America,Americas
5,Whitehorse,Canada,Americas
6,Beijing,China,Asia
7,Bombay,India,Asia
8,Sydney,Australia,Asia
9,Singapore,Singapore,Asia


16x3: 48 data elements


## Ejercicio w)
Realizar una relación con los nombres de los directores de los departamentos ordenado por apellidos de la Z a la A, esta única columna se llamará Nombre Completo, apareciendo los Apellidos una coma y el Nombre.

In [97]:
SQL("""
SELECT
    E.LAST_NAME || ', ' || E.FIRST_NAME AS "Nombre Completo",
    D.DEPARTMENT_NAME AS Departamento
FROM
    EMPLOYEES AS E,
    DEPARTMENTS AS D
WHERE
    E.EMPLOYEE_ID = D.MANAGER_ID
ORDER BY
    E.LAST_NAME DESC
;
""")

,Nombre Completo,Departamento
0,"Whalen, Jennifer",Administration
1,"Russell, John",Sales
2,"Raphaely, Den",Purchasing
3,"Mavris, Susan",Human Resources
4,"King, Steven",Executive
5,"Hunold, Alexander",IT
6,"Higgins, Shelley",Accounting
7,"Hartstein, Michael",Marketing
8,"Greenberg, Nancy",Finance
9,"Fripp, Adam",Shipping


11x2: 22 data elements


## Ejercicio x)
Obtener Apellidos, Salario y Nombre del departamento de aquellos empleados que, no siendo jefes de departamento, no superen el salario medio del departamento para el cual trabajan.

In [98]:
SQL("""
SELECT
    E.LAST_NAME AS Apellidos,
    E.SALARY AS Salario,
    D.DEPARTMENT_NAME AS Departamento
FROM
    EMPLOYEES AS E,
    DEPARTMENTS AS D
WHERE
    E.DEPARTMENT_ID = D.DEPARTMENT_ID
    AND
    E.EMPLOYEE_ID NOT IN (SELECT
        DISTINCT E2.MANAGER_ID
        FROM
            EMPLOYEES AS E2
        WHERE
            E2.MANAGER_ID IS NOT NULL
        )
    AND
    Salario < (SELECT
        AVG(E3.SALARY)
        FROM EMPLOYEES AS E3
        WHERE E3.DEPARTMENT_ID = E.DEPARTMENT_ID
        )
;
""")

,Apellidos,Salario,Departamento
0,Austin,4800,IT
1,Pataballa,4800,IT
2,Lorentz,4200,IT
3,Chen,8200,Finance
4,Sciarra,7700,Finance
...,...,...,...
58,Feeney,3000,Shipping
59,OConnell,2600,Shipping
60,Grant,2600,Shipping
61,Fay,6000,Marketing


63x3: 189 data elements


## Ejercicio y)
Realizar una lista de Nombre de Departamento y Ciudad donde éste se ubica de tal manera que ningún empleado trabaje en ellos y otra que sí tengan algún empleado.

In [99]:
SQL("""
SELECT
    D.DEPARTMENT_NAME AS Departamento,
    L.CITY AS Ciudad,
    "Sin Empleados" AS "Empleados?"
FROM
    DEPARTMENTS AS D,
    LOCATIONS AS L
WHERE
    D.LOCATION_ID = L.LOCATION_ID
    AND
    D.DEPARTMENT_ID NOT IN (SELECT
        E.DEPARTMENT_ID
        FROM
            EMPLOYEES AS E)
UNION
SELECT
    D.DEPARTMENT_NAME AS Departamento,
    L.CITY AS Ciudad,
    "Con Empleados" AS "Empleados?"
FROM
    DEPARTMENTS AS D,
    LOCATIONS AS L
WHERE
    D.LOCATION_ID = L.LOCATION_ID
    AND
    D.DEPARTMENT_ID IN (SELECT
        E.DEPARTMENT_ID
        FROM
            EMPLOYEES AS E)
""")

,Departamento,Ciudad,Empleados?
0,Accounting,Seattle,Con Empleados
1,Administration,Seattle,Con Empleados
2,Executive,Seattle,Con Empleados
3,Finance,Seattle,Con Empleados
4,Human Resources,London,Con Empleados
5,IT,Southlake,Con Empleados
6,Marketing,Toronto,Con Empleados
7,Public Relations,Munich,Con Empleados
8,Purchasing,Seattle,Con Empleados
9,Sales,Oxford,Con Empleados


11x3: 33 data elements


## Ejercicio z)
Obtener Nombre, Apellidos y Salario de los empleados cuyo salario sea mayor que el salario medio de los compañeros que trabajan es su mismo país.

In [100]:
SQL("""
SELECT
    E.FIRST_NAME AS Nombre,
    E.LAST_NAME AS Apellidos,
    E.SALARY AS Salario
FROM
    EMPLOYEES AS E,
    EMPLOYEES AS Empleado,
    DEPARTMENTS AS Emp_Dep,
    LOCATIONS AS Emp_Loc,
    COUNTRIES AS Emp_Countr
WHERE
    E.EMPLOYEE_ID = Empleado.EMPLOYEE_ID
    AND
    Empleado.DEPARTMENT_ID = Emp_Dep.DEPARTMENT_ID
    AND
    Emp_Dep.LOCATION_ID = Emp_Loc.LOCATION_ID
    AND
    Emp_Loc.COUNTRY_ID = Emp_Countr.COUNTRY_ID
    AND
    Empleado.SALARY > (SELECT
        AVG(Companeros.SALARY)
        FROM
            EMPLOYEES AS Companeros,
            DEPARTMENTS AS Comp_Dep,
            LOCATIONS AS Comp_Loc,
            COUNTRIES AS Comp_Countr
        WHERE
            Companeros.EMPLOYEE_ID <> Empleado.EMPLOYEE_ID
            AND
            Companeros.DEPARTMENT_ID = Comp_Dep.DEPARTMENT_ID
            AND
            Comp_Dep.LOCATION_ID = Comp_Loc.LOCATION_ID
            AND
            Comp_Loc.COUNTRY_ID = Comp_Countr.COUNTRY_ID
            AND
            Emp_Countr.COUNTRY_ID = Comp_Countr.COUNTRY_ID
        )
;
""")

,Nombre,Apellidos,Salario
0,Steven,King,24000
1,Neena,Kochhar,17000
2,Lex,De Haan,17000
3,Alexander,Hunold,9000
4,Bruce,Ernst,6000
5,Nancy,Greenberg,12000
6,Daniel,Faviet,9000
7,John,Chen,8200
8,Ismael,Sciarra,7700
9,Jose Manuel,Urman,7800


37x3: 111 data elements


## Ejercicio aa)
Obtener por cada tipo de trabajo (JOB_ID), indicando también su nombre (JOB_TITLE), el salario más bajo pagado (es decir cuál es el importe pagado a aquel empleado que menos gana).

In [101]:
SQL("""
SELECT
        j.job_id AS "Tipo de Trabajo",
        j.job_title AS "Puesto",
        min(e.salary) AS "Menor Salario"
    From
        JOBS as j,
        EMPLOYEES as e
    where
        j.job_id = e.job_id
    group by j.job_id
    """)

,Tipo de Trabajo,Puesto,Menor Salario
0,AC_ACCOUNT,Public Accountant,8300
1,AC_MGR,Accounting Manager,12000
2,AD_ASST,Administration Assistant,4400
3,AD_PRES,President,24000
4,AD_VP,Administration Vice President,17000
5,FI_ACCOUNT,Accountant,6900
6,FI_MGR,Finance Manager,12000
7,HR_REP,Human Resources Representative,6500
8,IT_PROG,Programmer,4200
9,MK_MAN,Marketing Manager,13000


19x3: 57 data elements


## Ejercicio bb)
Listar el nombre, apellidos y salario de los empleados cuyo salario sea mayor que el salario máximo de al menos uno de los departamentos situados en su mismo país.

In [102]:
SQL("""
SELECT
    E.first_name  AS Nombre,
    E.last_name   AS Apellidos,
    E.salary      AS Salario
FROM
    EMPLOYEES   AS E,
    DEPARTMENTS AS D,
    LOCATIONS   AS L,
    COUNTRIES   AS C
WHERE
    E.department_id   = D.department_id
    AND D.location_id = L.location_id
    AND L.country_id  = C.country_id

    AND EXISTS (
        SELECT 1
        FROM
            EMPLOYEES   AS E2,
            DEPARTMENTS AS D2,
            LOCATIONS   AS L2,
            COUNTRIES   AS C2
        WHERE
            E2.department_id  = D2.department_id
            AND D2.location_id = L2.location_id
            AND L2.country_id = C2.country_id
            AND C2.country_id = C.country_id
        GROUP BY
            D2.department_id
        HAVING
            MAX(E2.salary) < E.salary
    )
;


""")

,Nombre,Apellidos,Salario
0,Steven,King,24000
1,Neena,Kochhar,17000
2,Lex,De Haan,17000
3,Alexander,Hunold,9000
4,Bruce,Ernst,6000
5,David,Austin,4800
6,Valli,Pataballa,4800
7,Nancy,Greenberg,12000
8,Daniel,Faviet,9000
9,John,Chen,8200


51x3: 153 data elements


## Ejercicio cc)
Obtener Nombre, Apellidos, Salario y Nombre del departamento, de aquellos empleados que superen el 75% del salario del jefe de su departamento.

1. Ρ:
    * $E:= ρ_{E}(EMPLOYEES)$
    * $D:= ρ_{D}(DEPARTMENTS)$
    * $S:= ρ_{S}(EMPLOYEES)$
3. $Α := (E.departmentId = D.departmentId)$
4. $Β := (D.managerId = S.employeeId)$
5. $Γ := (E.salary > 0.75 · S.salary)$
7. $(E ⨝_Α D)⨝_Β S$
   1. $R_1 := σ_{(Α ∧ Β)}[E ⨯ D ⨯ S]$
7. $R_2 := σ(Γ)[R_1]$
8. ⬦ $π(E.firstName, E.lastName, E.salary, D.departmentName)[R_2]$





In [103]:
SQL("""
SELECT
    e.first_name AS Nombre,
    e.last_name AS Apellidos,
    e.salary AS Salario,
    d.department_name AS Departamento
FROM
    EMPLOYEES AS e,
    DEPARTMENTS AS d,
    EMPLOYEES AS j
WHERE
    e.department_id = d.department_id
    AND
    d.manager_id = j.employee_id
    AND
    e.salary > j.salary * 0.75
""")

,Nombre,Apellidos,Salario,Departamento
0,Steven,King,24000,Executive
1,Alexander,Hunold,9000,IT
2,Nancy,Greenberg,12000,Finance
3,Den,Raphaely,11000,Purchasing
4,Matthew,Weiss,8000,Shipping
5,Adam,Fripp,8200,Shipping
6,Payam,Kaufling,7900,Shipping
7,Shanta,Vollman,6500,Shipping
8,John,Russell,14000,Sales
9,Karen,Partners,13500,Sales


19x4: 76 data elements


## Ejercicio dd)
Obtener Apellidos, Salario y Nombre del departamento de aquellos empleados que trabajen en un departamento alojado en una ciudad en la cual estén alojados otros departamentos de la empresa (lo cual significa que en esa ciudad habrá al menos dos departamentos alojados) y además no superen el salario medio del departamento para el cual trabajan.

In [104]:
SQL("""
SELECT
    E.last_name       AS Apellidos,
    E.salary          AS Salario,
    D.department_name AS Departamento
FROM
    EMPLOYEES   AS E,
    DEPARTMENTS AS D,
    LOCATIONS   AS L,
    (
        SELECT
            L2.city AS City
        FROM
            DEPARTMENTS AS D2,
            LOCATIONS   AS L2
        WHERE
            D2.location_id = L2.location_id
        GROUP BY
            L2.city
        HAVING
            COUNT(*) > 1
    ) AS Ciudades_Validas
WHERE
    E.department_id = D.department_id
    AND D.location_id = L.location_id
    AND L.city = Ciudades_Validas.City
    AND E.salary <= (
        SELECT AVG(EmpD.salary)
        FROM EMPLOYEES AS EmpD
        WHERE EmpD.department_id = E.department_id
        )
;
""")


,Apellidos,Salario,Departamento
0,Kochhar,17000,Executive
1,De Haan,17000,Executive
2,Chen,8200,Finance
3,Sciarra,7700,Finance
4,Urman,7800,Finance
5,Popp,6900,Finance
6,Khoo,3100,Purchasing
7,Baida,2900,Purchasing
8,Tobias,2800,Purchasing
9,Himuro,2600,Purchasing


13x3: 39 data elements


## Ejercicio ee)
Obtener una lista de empleados incluyendo una columna con su nombre y apellidos (estos dos campos serán una cadena única estando separados por una coma) y otra con el número de compañeros que tengan el mismo trabajo que éste. Esta lista debe ordenarse alfabéticamente según el nombre de departamento al cual pertenezca, de la “Z” a la “A”. Esta lista solo afectará a empleados asignados a algún departamento.

In [105]:
SQL("""
SELECT
    E.LAST_NAME || ', ' || E.FIRST_NAME AS "Nombre Completo",
    (SELECT
        COUNT(*)
        FROM
            EMPLOYEES AS E2
        WHERE
            E2.JOB_ID = E.JOB_ID
            )-1 AS "Número de compañeros"
FROM
    EMPLOYEES AS E,
    DEPARTMENTS AS D
WHERE
    E.DEPARTMENT_ID = D.DEPARTMENT_ID
ORDER BY
    D.DEPARTMENT_NAME DESC
;
""")

,Nombre Completo,Número de compañeros
0,"Weiss, Matthew",4
1,"Fripp, Adam",4
2,"Kaufling, Payam",4
3,"Vollman, Shanta",4
4,"Mourgos, Kevin",4
...,...,...
101,"Kochhar, Neena",1
102,"De Haan, Lex",1
103,"Whalen, Jennifer",0
104,"Higgins, Shelley",0


106x2: 212 data elements


## Ejercicio ff)
Dar una lista con el nombre y apellidos de los empleados cuyo trabajo actual tenga un salario máximo menor a 12000 €, incluyendo el nombre y apellidos de su supervisor. Esta lista solo debe incluir a empleados pertenecientes a algún departamento, cuyo salario (del empleado en cuestión) sea menor que el salario promedio de todos sus compañeros de departamento con igual trabajo que éste.

In [106]:
SQL("""
SELECT
    E.LAST_NAME AS Apellidos,
    E.FIRST_NAME AS Nombre,
    S.LAST_NAME AS Apellidos_Supervisor,
    S.FIRST_NAME AS Nombre_Supervisor
FROM
    EMPLOYEES AS E,
    EMPLOYEES AS S,
    JOBS AS J
WHERE
    E.MANAGER_ID = S.EMPLOYEE_ID
    AND
    E.JOB_ID = J.JOB_ID
    AND
    J.MAX_SALARY < 12000
    AND
    E.DEPARTMENT_ID IS NOT NULL
    AND
    E.SALARY < ( SELECT
        AVG(C.SALARY)
        FROM EMPLOYEES AS C
        WHERE
            C.DEPARTMENT_ID = E.DEPARTMENT_ID
            AND
            C.JOB_ID = E.JOB_ID
            AND
            E.EMPLOYEE_ID <> C.EMPLOYEE_ID

            )


""")

,Apellidos,Nombre,Apellidos_Supervisor,Nombre_Supervisor
0,Austin,David,Hunold,Alexander
1,Pataballa,Valli,Hunold,Alexander
2,Lorentz,Diana,Hunold,Alexander
3,Sciarra,Ismael,Greenberg,Nancy
4,Urman,Jose Manuel,Greenberg,Nancy
5,Popp,Luis,Greenberg,Nancy
6,Himuro,Guy,Raphaely,Den
7,Colmenares,Karen,Raphaely,Den
8,Vollman,Shanta,King,Steven
9,Mourgos,Kevin,King,Steven


34x4: 136 data elements


## Ejercicio gg)
Proveer de una lista de empleados que sean supervisores indicando Nombre y Apellidos de éste que incluya el número de empleados no directores de departamento que dicho supervisor supervisa, el número de empleados directores de departamento que dicho supervisor supervisa. De estas dos últimas columnas se debe excluir empleados que sean supervisores.

> Si vemos el problema por partes, se puede resolver utilizando tablas intermedias. Una de ellas debería de identificar los Subalternos de cada Supervisor. La otra, los directores Supervisados.

In [107]:
SQL("""
    /* Subordinados NO directores */
    SELECT
        M.employee_id AS Id,
        COUNT(*)      AS NumSub,
        0             AS NumDir
    FROM
        EMPLOYEES AS M,
        EMPLOYEES AS E
    WHERE
        M.employee_id = E.manager_id
        AND E.employee_id NOT IN (
            SELECT Dep.manager_id
            FROM DEPARTMENTS AS Dep
            WHERE Dep.manager_id IS NOT NULL
        )
    GROUP BY
        M.employee_id

    UNION ALL

    /* Subordinados que SÍ son directores */
    SELECT
        M.employee_id AS Id,
        0             AS NumSub,
        COUNT(*)      AS NumDir
    FROM
        EMPLOYEES AS M,
        EMPLOYEES AS E
    WHERE
        M.employee_id = E.manager_id
        AND E.employee_id IN (
            SELECT Dep.manager_id
            FROM DEPARTMENTS AS Dep
            WHERE Dep.manager_id IS NOT NULL
        )
    GROUP BY
        M.employee_id
""")


,Id,NumSub,NumDir
0,100,10,0
1,103,4,0
2,108,5,0
3,114,5,0
4,120,8,0
5,121,8,0
6,122,8,0
7,123,8,0
8,124,8,0
9,145,6,0


19x3: 57 data elements


In [108]:
SQL("""
SELECT
    Super.last_name  AS "Apellido Supervisor",
    Super.first_name AS "Nombre Supervisor",
    SUM(Supervisados.Sub) AS "Nº Subordinados",
    SUM(Supervisados.Dir) AS "Nº Directores Supervisados"
FROM
    EMPLOYEES AS Super,
    (
        SELECT
            M.employee_id AS Id,
            COUNT(*)      AS Sub,
            0             AS Dir
        FROM EMPLOYEES AS M, EMPLOYEES AS E
        WHERE
            M.employee_id = E.manager_id
            AND E.employee_id NOT IN (
                SELECT Dep.manager_id
                FROM DEPARTMENTS AS Dep
                WHERE Dep.manager_id IS NOT NULL
            )
        GROUP BY M.employee_id

        UNION ALL

        SELECT
            M.employee_id AS Id,
            0             AS Sub,
            COUNT(*)      AS Dir
        FROM EMPLOYEES AS M, EMPLOYEES AS E
        WHERE
            M.employee_id = E.manager_id
            AND E.employee_id IN (
                SELECT Dep.manager_id
                FROM DEPARTMENTS AS Dep
                WHERE Dep.manager_id IS NOT NULL
            )
        GROUP BY M.employee_id
    ) AS Supervisados
WHERE
    Super.employee_id = Supervisados.Id
GROUP BY
    Super.employee_id, Super.last_name, Super.first_name
ORDER BY
    Super.last_name, Super.first_name
;


""")

,Apellido Supervisor,Nombre Supervisor,Nº Subordinados,Nº Directores Supervisados
0,Cambrault,Gerald,6,0
1,De Haan,Lex,0,1
2,Errazuriz,Alberto,6,0
3,Fripp,Adam,8,0
4,Greenberg,Nancy,5,0
5,Hartstein,Michael,1,0
6,Higgins,Shelley,1,0
7,Hunold,Alexander,4,0
8,Kaufling,Payam,8,0
9,King,Steven,10,4


18x4: 72 data elements


## Ejercicio hh)
Realizar una lista de directores de departamento con sus nombres y apellidos, incluyendo: el número de empleados pertenecientes al departamento que dirige y que son supervisores, el sueldo medio de todos los empleados de su departamento que no son supervisores. Esta lista sólo debe incluir directores que superen en un 10% el salario promedio del resto de directores de departamento de la empresa que no sean supervisores.

He explorado como hacerla con tablas intermedias:

In [109]:
# Supervisores
SQL("""
SELECT DISTINCT
    E2.MANAGER_ID AS supervisor_id
FROM EMPLOYEES AS E2
WHERE E2.MANAGER_ID IS NOT NULL

""")

,supervisor_id
0,100.0
1,102.0
2,103.0
3,101.0
4,108.0
5,114.0
6,120.0
7,121.0
8,122.0
9,123.0


18x1: 18 data elements


In [110]:
# Supervisores también directores
SQL("""
SELECT
    D.MANAGER_ID   AS director_id,
    COUNT(*)       AS num_supervisores
FROM DEPARTMENTS AS D,
     EMPLOYEES   AS E
WHERE
    D.DEPARTMENT_ID = E.DEPARTMENT_ID
    AND E.EMPLOYEE_ID IN (
        SELECT DISTINCT
            E2.MANAGER_ID
        FROM EMPLOYEES AS E2
        WHERE E2.MANAGER_ID IS NOT NULL
    )
GROUP BY
    D.MANAGER_ID
""")

,director_id,num_supervisores
0,100.0,3
1,103.0,1
2,108.0,1
3,114.0,1
4,121.0,5
5,145.0,5
6,201.0,1
7,205.0,1


8x2: 16 data elements


In [111]:
# Directores
SQL("""
SELECT
    D.MANAGER_ID   AS director_id,
    D.DEPARTMENT_ID
FROM DEPARTMENTS AS D
WHERE D.MANAGER_ID IS NOT NULL
""")

,director_id,department_id
0,200.0,10
1,201.0,20
2,114.0,30
3,203.0,40
4,121.0,50
5,103.0,60
6,204.0,70
7,145.0,80
8,100.0,90
9,108.0,100


11x2: 22 data elements


Final

In [112]:
SQL("""
SELECT
    Dir.first_name,
    Dir.last_name,
    Sup.num_supervisores,
    AvgNS.avg_salary_nosuper
FROM
    EMPLOYEES AS Dir,

    (
        SELECT
            D.MANAGER_ID   AS director_id,
            COUNT(*)       AS num_supervisores
        FROM DEPARTMENTS AS D,
             EMPLOYEES   AS E
        WHERE
            D.DEPARTMENT_ID = E.DEPARTMENT_ID
            AND E.EMPLOYEE_ID IN (
                SELECT DISTINCT
                    E2.MANAGER_ID
                FROM EMPLOYEES AS E2
                WHERE E2.MANAGER_ID IS NOT NULL
            )
        GROUP BY
            D.MANAGER_ID
    ) AS Sup,

    (
        SELECT
            D.MANAGER_ID   AS director_id,
            AVG(E.SALARY)  AS avg_salary_nosuper
        FROM DEPARTMENTS AS D,
             EMPLOYEES   AS E
        WHERE
            D.DEPARTMENT_ID = E.DEPARTMENT_ID
            AND E.EMPLOYEE_ID NOT IN (
                SELECT DISTINCT
                    E2.MANAGER_ID
                FROM EMPLOYEES AS E2
                WHERE E2.MANAGER_ID IS NOT NULL
            )
        GROUP BY
            D.MANAGER_ID
    ) AS AvgNS

WHERE
    Dir.EMPLOYEE_ID = Sup.director_id
    AND Dir.EMPLOYEE_ID = AvgNS.director_id
    AND Dir.SALARY > 1.10 * (
        SELECT
            AVG(E.SALARY)
        FROM EMPLOYEES AS E
        WHERE
            E.EMPLOYEE_ID IN (
                SELECT
                    D.MANAGER_ID
                FROM DEPARTMENTS AS D
                WHERE D.MANAGER_ID IS NOT NULL
            )
            AND E.EMPLOYEE_ID NOT IN (
                SELECT DISTINCT
                    E2.MANAGER_ID
                FROM EMPLOYEES AS E2
                WHERE E2.MANAGER_ID IS NOT NULL
            )
    )
ORDER BY
    Dir.last_name,
    Dir.first_name
;


""")

,first_name,last_name,num_supervisores,avg_salary_nosuper
0,Adam,Fripp,5,3000.000000
1,Nancy,Greenberg,1,7920.000000
2,Michael,Hartstein,1,6000.000000
3,Shelley,Higgins,1,8300.000000
4,Alexander,Hunold,1,4950.000000
5,Den,Raphaely,1,2780.000000
6,John,Russell,5,8396.551724


7x4: 28 data elements
